# [A100 40GB] ViRL39K 라운드 GRPO LoRA → MMMU validation 900문항

`MMDL/virl39k/train/ViRL_RoundGRPO_4B_A100_Train_v4.ipynb`가 ViRL 검증으로 고른 adapter(`eval_handoff.json`)를 **공식 baseline과 같은 A100-SXM4-40GB**에서 평가합니다. 평가 코드·프롬프트·설정·시드·파서는 baseline·제출 모델 평가와 같고, 바뀌는 것은 **모델 출처(adapter를 이 노트북에서 병합), 출력 폴더, 마지막 판정 셀**뿐입니다.

- 비교 대상: baseline **575/900 (63.89%, 40GB)**, 제출 모델 **569/900 (63.22%, 40GB)**.
- GRPO 학습은 원래 base 4B에서 시작했습니다(제출 모델 SFT 위가 아님). 그래서 baseline 대비 변화가 곧 GRPO 효과입니다.
- **학습 노트북이 base를 골랐으면(ViRL 검증 개선 없음) 0번 셀에서 멈춥니다.** 사전 규칙상 MMMU 평가를 하지 않습니다(유닛 절약).
- **40GB가 아닌 GPU가 잡히면 3번 셀에서 멈춥니다.** 런타임 연결 해제 후 다시 연결해 40GB를 받으세요.

**채택 규칙 (평가 전에 고정, 2026-11-19 짧은 풀이 실험과 같음):**
1. 첫 평가(seed 3407)가 **594/900 미만**이면 **제출 모델 유지**. 재평가하지 않습니다.
2. **594 이상**이면 같은 adapter·설정·40GB로 **확인 재평가**(seed 42, `CONFIRM_RUN = True`).
3. **두 평가 평균 588 이상**일 때만 채택. 재시도 없음.
4. 보고서에는 제출 모델 이후 평가한 변형(A-전체 550, A-소량 562, 짧은 풀이 571, 이번 GRPO)과 이 규칙을 함께 적습니다.
5. **v4 2라운드 step 170은 사후 추가 후보입니다**(2라운드가 step 175에서 중단돼 원래 선택은 base). ViRL 검증에서 같은 기준(base+12, 출력 길이)을 통과해 평가합니다. ViRL 검증으로 후보를 여러 개 본 뒤라 확인 재평가 규칙이 특히 중요하고, 보고서에 이 경위를 적습니다.

> **실행 전:** 런타임 유형 **A100 GPU**, 40GB 확인. adapter 병합 약 5~10분 + 900문항 생성 약 2시간 = **약 27~30유닛**. 세션이 끊기면 0번 셀부터 다시 실행하면 병합과 끝난 과목은 건너뜁니다. **5번 셀(추론)이 도는 동안 다른 셀을 실행하지 마세요.**


## 0. Drive 연결, 학습 결과(handoff) 확인

- **`GRPO_RUN_TAG`**: 학습 노트북의 `RUN_TAG`(기본 `round_grpo_v4`).
- 학습 노트북이 쓴 `eval_handoff.json`에서 고른 adapter(라운드)를 읽고, adapter 파일 해시가 handoff의 `policy_id`와 같은지 확인합니다(학습 뒤 바뀐 파일을 평가하지 않도록).
- **`CONFIRM_RUN`**: 처음에는 `False`(seed 3407). 7-2 셀이 "확인 재평가 필요"라고 할 때만 `True`로 바꿔 이 셀부터 다시 실행(seed 42). 확인 재평가는 첫 평가와 **같은 adapter(policy_id)**여야 진행합니다.
- 결과는 새 폴더(`virl39k_grpo_<RUN_TAG>_<라운드>_seed<SEED>_16k_a100_40gb`)에 저장합니다.


In [1]:
from google.colab import drive
from pathlib import Path
import os, sys, json, re, subprocess, shutil, hashlib, datetime, time

drive.mount('/content/drive')

# === 사용자 설정 ===
VARIANT = 'grpo'          # 기록용 (바꾸지 마세요)
GRPO_RUN_TAG = 'round_grpo_v4'   # 학습 노트북의 RUN_TAG
POSTHOC_CANDIDATE = 'round_02_step170'   # 사후 검증(posthoc_verdict.json)을 통과한 후보. None이면 eval_handoff.json의 선택을 씀
CONFIRM_RUN = False       # 7-2 셀이 '확인 재평가 필요'라고 할 때만 True (seed 42로 한 번 더 평가)
EVAL_SEED = 42 if CONFIRM_RUN else 3407
ADAPTER_PCT = None        # GRPO는 라운드 checkpoint를 씀 (기록용)

DRIVE_EVAL_PROJECT = Path('/content/drive/MyDrive/MMDL/code/qwen3_vl_mmmu_eval')
GRPO_RUN_DIR = Path('/content/drive/MyDrive/VIRL39K_RFT_4B') / GRPO_RUN_TAG
HANDOFF_FILE = GRPO_RUN_DIR / 'eval_handoff.json'
BASE_MODEL_ID = 'Qwen/Qwen3-VL-4B-Instruct'
BASE_REVISION = 'ebb281ec70b05090aa6165b016eac8ec08e71b17'

# --- 학습 결과(handoff) 확인 ---
assert HANDOFF_FILE.is_file(), f'학습 노트북의 eval_handoff.json이 없습니다: {HANDOFF_FILE}. full 학습이 끝났는지 확인하세요.'
handoff = json.loads(HANDOFF_FILE.read_text())
assert handoff['model'] == BASE_MODEL_ID and handoff['revision'] == BASE_REVISION, f'base 모델이 다릅니다: {handoff}'
if POSTHOC_CANDIDATE:
    # 2026-11-21: v4 2라운드가 step 175에서 중단돼 handoff는 base. step 170 checkpoint를 같은 ViRL 기준(base+12, 길이)으로
    # 사후 검증해 통과한 경우에만 평가한다(사후 추가 후보 → 보고서에 명시).
    pv_file = GRPO_RUN_DIR / 'validation' / POSTHOC_CANDIDATE / 'posthoc_verdict.json'
    assert pv_file.is_file(), f'사후 검증 결과가 없습니다: {pv_file}'
    pv = json.loads(pv_file.read_text())
    assert pv['candidate'] == POSTHOC_CANDIDATE and pv['passed'], f"사후 검증을 통과하지 않은 후보입니다: {pv.get('decision')}"
    handoff = dict(handoff, selected_name=pv['candidate'], selected_adapter=pv['adapter'], policy_id=pv['policy_id'],
                   proxy_gain=pv['report']['correct'] - pv['baseline']['correct'],
                   virl_validation=pv['report'], baseline_virl_validation=pv['baseline'], posthoc=True)
    print('사후 추가 후보:', POSTHOC_CANDIDATE, '| ViRL 검증', pv['report']['correct'], 'vs base', pv['baseline']['correct'],
          '| 문항별', pv['paired_vs_base'])
SELECTED_NAME = handoff['selected_name']
SELECTED_ADAPTER = handoff['selected_adapter']
print('ViRL 검증 선택:', SELECTED_NAME, '| proxy gain(검증 샘플 정답 수):', handoff['proxy_gain'])
print('  base:', {k: handoff['baseline_virl_validation'].get(k) for k in ('n', 'correct', 'accuracy', 'truncated_fraction', 'output_tokens_median')})
print('  선택:', {k: handoff['virl_validation'].get(k) for k in ('n', 'correct', 'accuracy', 'truncated_fraction', 'output_tokens_median')})
assert SELECTED_ADAPTER is not None and handoff['proxy_gain'] > 0, (
    'ViRL 검증에서 base보다 나아진 라운드가 없습니다(학습 노트북이 base를 선택). '
    '사전 규칙상 MMMU 평가를 하지 않고 제출 모델을 유지합니다.')
ADAPTER_DIR = Path(SELECTED_ADAPTER)
assert ADAPTER_DIR.is_dir() and (ADAPTER_DIR/'adapter_model.safetensors').is_file(), f'adapter 폴더가 없습니다: {ADAPTER_DIR}'
def _sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(2**20), b''): h.update(b)
    return h.hexdigest()
POLICY_ID = hashlib.sha256(json.dumps({n: _sha(ADAPTER_DIR/n) for n in ('adapter_config.json', 'adapter_model.safetensors')},
                                      sort_keys=True, ensure_ascii=False, separators=(',', ':')).encode()).hexdigest()
assert POLICY_ID == handoff['policy_id'], 'adapter 파일이 학습 노트북이 고른 것과 다릅니다(해시 불일치).'
print('✔ adapter 해시 = handoff policy_id', POLICY_ID[:16], '|', ADAPTER_DIR)

# MMMU 데이터 캐시 (이전 평가 때 만든 Drive 캐시 재사용)
DATA_ROOT = Path('/content/drive/MyDrive/MMDL/pmc_vqa/eval/mmmu_dataset_cache')

# 결과 저장 (실험마다 독립 폴더)
DRIVE_RESULTS_BASE = Path('/content/drive/MyDrive/MMDL/pmc_vqa/eval/mmmu_eval_outputs')
REQUIRED_GPU = 'NVIDIA A100-SXM4-40GB'   # 공식 baseline·제출 모델과 같은 GPU 변형 (MMMU 비교는 같은 GPU에서만)
EXPERIMENT_NAME = f'virl39k_grpo_{GRPO_RUN_TAG}_{SELECTED_NAME}_seed{EVAL_SEED}_16k_a100_40gb'
FIRST_V2_DIR = DRIVE_RESULTS_BASE / f'virl39k_grpo_{GRPO_RUN_TAG}_{SELECTED_NAME}_seed3407_16k_a100_40gb_v2'   # 첫 평가(seed 3407) 판정 위치
# --- 확인 재평가 관문 (사전 규칙: 첫 평가 >= 594일 때만, 같은 adapter만) ---
if CONFIRM_RUN:
    fv = FIRST_V2_DIR / 'grpo_verdict.json'
    assert fv.is_file(), f'첫 평가(seed 3407) 판정 파일이 없습니다: {fv}. CONFIRM_RUN=False로 첫 평가부터 하세요.'
    first_verdict = json.loads(fv.read_text())
    assert first_verdict['passes_594'], f"첫 평가가 {first_verdict['new_correct']}/900으로 594 미만입니다. 규칙상 확인 재평가를 하지 않습니다(제출 모델 유지)."
    assert first_verdict.get('policy_id') == POLICY_ID, '첫 평가와 다른 adapter입니다. 확인 재평가는 같은 adapter로만 합니다.'
    print('확인 재평가: 첫 평가', first_verdict['new_correct'], '/900 → seed 42로 다시 평가합니다')
# --- 확인 재평가 관문 끝 ---
OUTPUTS_ROOT = DRIVE_RESULTS_BASE / EXPERIMENT_NAME
RUN_NAME = 'lora_merged'
RUN_DIR = OUTPUTS_ROOT / RUN_NAME
V2_REPORT_DIR = DRIVE_RESULTS_BASE / (EXPERIMENT_NAME + '_v2')

# 비교 대상 (없으면 비교만 건너뜀)
BASELINE_RUN_DIR = None   # None이면 아래 후보에서 자동 탐색
PREV_RUN_DIR = DRIVE_RESULTS_BASE / 'virl39k_100pct_seed3407_16k_a100_40gb' / 'lora_merged'   # 제출 모델 40GB 평가(569/900)
assert OUTPUTS_ROOT != PREV_RUN_DIR.parent, '출력 폴더가 제출 모델 평가와 같습니다'

# adapter를 이 노트북에서 병합해 Colab 로컬에 둡니다(Drive에 9GB를 쓰지 않음). 재개할 때도 같은 경로.
LOCAL_MODEL = Path(f'/content/virl39k_grpo_{GRPO_RUN_TAG}_{SELECTED_NAME}_{POLICY_ID[:12]}_merged_for_eval')
EVAL_VENV = Path('/content/mmmu_eval_venv')

def scored_count(run_dir):
    f = run_dir / 'scored.jsonl'
    return sum(1 for l in open(f, encoding='utf-8') if l.strip()) if f.is_file() else 0

RESUME = (RUN_DIR / 'run_meta.json').is_file()
ALREADY_DONE = scored_count(RUN_DIR) == 900
if ALREADY_DONE:
    print('이미 900문항 생성·채점 완료 → 생성 단계는 건너뜁니다:', RUN_DIR)
elif RESUME:
    print('이전 실행 기록이 있습니다 → 끝난 과목은 건너뛰고 이어서 생성합니다:', RUN_DIR)
else:
    if RUN_DIR.is_dir() and {p.name for p in RUN_DIR.iterdir()} <= {'.run.lock'}:
        running = subprocess.run(['pgrep', '-f', 'mmmu_eval[.]cli'], capture_output=True).returncode == 0
        assert not running, f'평가 프로세스가 아직 돌고 있습니다. !pkill -f mmmu_eval.cli 로 종료한 뒤 다시 실행하세요: {RUN_DIR}'
        shutil.rmtree(RUN_DIR)
        print('이전 시도가 시작 직후 끊겨 잠금 파일만 남은 폴더를 지웠습니다:', RUN_DIR)
    assert not RUN_DIR.exists() or not any(RUN_DIR.iterdir()), f'run_meta.json 없이 파일만 있는 폴더입니다(지우지 말고 내용을 확인하세요): {RUN_DIR}'
    print('새 실행:', RUN_DIR)
print('평가 변형:', VARIANT, GRPO_RUN_TAG, SELECTED_NAME, '| seed', EVAL_SEED, '| 요구 GPU:', REQUIRED_GPU)
print('제출 모델 40GB 평가(비교용):', PREV_RUN_DIR, '있음' if (PREV_RUN_DIR/'scored.jsonl').is_file() else '⚠ 없음(비교 셀 일부 건너뜀)')


Mounted at /content/drive
사후 추가 후보: round_02_step170 | ViRL 검증 910 vs base 894 | 문항별 {'better': 42, 'worse': 31, 'same': 223, 'sign_test_p_two_sided': 0.24164472314075197, 'net_samples': 16}
ViRL 검증 선택: round_02_step170 | proxy gain(검증 샘플 정답 수): 16
  base: {'n': 1184, 'correct': 894, 'accuracy': 0.7550675675675675, 'truncated_fraction': 0.06756756756756757, 'output_tokens_median': 501.0}
  선택: {'n': 1184, 'correct': 910, 'accuracy': 0.768581081081081, 'truncated_fraction': 0.059966216216216214, 'output_tokens_median': 492.5}
✔ adapter 해시 = handoff policy_id 55922b8dfc2948d1 | /content/drive/MyDrive/VIRL39K_RFT_4B/round_grpo_v4/round_02/train/checkpoint_00680_00170
새 실행: /content/drive/MyDrive/MMDL/pmc_vqa/eval/mmmu_eval_outputs/virl39k_grpo_round_grpo_v4_round_02_step170_seed3407_16k_a100_40gb/lora_merged
평가 변형: grpo round_grpo_v4 round_02_step170 | seed 3407 | 요구 GPU: NVIDIA A100-SXM4-40GB
제출 모델 40GB 평가(비교용): /content/drive/MyDrive/MMDL/pmc_vqa/eval/mmmu_eval_outputs/virl39k_100pct_se

## 1. Drive의 기존 MMMU 평가 코드 확인

외부 저장소를 가져오지 않고 `DRIVE_EVAL_PROJECT`를 그대로 씁니다. 설정·스크립트 SHA-256을 기록합니다. baseline 결과 폴더도 함께 찾습니다.

In [2]:
candidates = [DRIVE_EVAL_PROJECT, DRIVE_EVAL_PROJECT/'code'/'qwen3_vl_mmmu_eval', DRIVE_EVAL_PROJECT/'qwen3_vl_mmmu_eval']
PROJECT = next((p for p in candidates if (p/'configs'/'final_16k.yaml').is_file()
                and (p/'scripts'/'run_mmmu_eval.sh').is_file()), None)
if PROJECT is None:
    raise FileNotFoundError('Drive 평가 코드 경로가 다릅니다. DRIVE_EVAL_PROJECT를 수정하세요.\n'
                            f'검사한 위치: {[str(c) for c in candidates]}')
CONFIG = PROJECT/'configs'/'final_16k.yaml'
EVAL_SH = PROJECT/'scripts'/'run_mmmu_eval.sh'
RESCORE_PY = PROJECT/'scripts'/'rescore.py'
REQ = PROJECT/'requirements.txt'
for p in (CONFIG, EVAL_SH, RESCORE_PY, REQ, PROJECT/'src'/'mmmu_eval'/'parse_v2.py'):
    assert p.is_file(), f'필수 평가 파일 없음: {p}'
print('✔ Drive 평가 프로젝트:', PROJECT)
for p in (CONFIG, EVAL_SH, RESCORE_PY):
    print(f'  {p.relative_to(PROJECT)} | sha256={hashlib.sha256(p.read_bytes()).hexdigest()}')

if BASELINE_RUN_DIR is None:
    base_candidates = [PROJECT/'outputs'/'final_seed3407_16k',
                       Path('/content/drive/MyDrive/MMDL/code/qwen3_vl_mmmu_eval/outputs/final_seed3407_16k')]
    BASELINE_RUN_DIR = next((p for p in base_candidates if (p/'scored.jsonl').is_file()
                             and (p/'resolved_config.json').is_file()), None)
if BASELINE_RUN_DIR is None:
    print('⚠ baseline scored.jsonl을 찾지 못했습니다. 평가는 진행되지만 baseline 비교는 건너뜁니다.')
else:
    print('✔ baseline 결과:', BASELINE_RUN_DIR, f'({scored_count(BASELINE_RUN_DIR)}문항)')

✔ Drive 평가 프로젝트: /content/drive/MyDrive/MMDL/code/qwen3_vl_mmmu_eval
  configs/final_16k.yaml | sha256=a6b0ac30c57e5d2a4358233060cd45ff642a3ba2193eeeecbeea45486334d818
  scripts/run_mmmu_eval.sh | sha256=81cb7d9bddfe18b8baf741fc5f7e2492d8f98656d474a1f82e1acd9677daffc3
  scripts/rescore.py | sha256=170c649cc98bf8df1a3dfeac9b4da7cef39904d13ce90119eb82fca6d97c1934
✔ baseline 결과: /content/drive/MyDrive/MMDL/code/qwen3_vl_mmmu_eval/outputs/final_seed3407_16k (900문항)


## 2. 평가 전용 가상환경 설치 (Python 3.12, baseline과 같은 버전)

Drive의 `requirements.txt`를 **별도 가상환경**에 설치합니다. Colab 기본 환경은 건드리지 않으므로 런타임 재시작이 필요 없습니다. 처음 한 번은 몇 분 걸리고, 같은 세션에서 다시 실행하면 바로 넘어갑니다.

평가 스크립트는 `PYTHON_BIN` 환경변수로 이 가상환경의 Python을 사용합니다.

In [3]:
def checked(command, *, cwd=None, env=None):
    print('$', ' '.join(map(str, command)), flush=True)
    return subprocess.run([str(v) for v in command], cwd=cwd, env=env, check=True)

EVAL_PY = EVAL_VENV/'bin'/'python'
marker = EVAL_VENV/'.installed_ok'
req_hash = hashlib.sha256(REQ.read_bytes()).hexdigest()
if not (marker.is_file() and marker.read_text() == req_hash):
    shutil.rmtree(EVAL_VENV, ignore_errors=True)
    checked([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
    checked([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', str(EVAL_VENV)])
    print('requirements:\n' + REQ.read_text(encoding='utf-8'))
    checked([sys.executable, '-m', 'uv', 'pip', 'install', '--python', str(EVAL_PY), '-r', str(REQ)])
    marker.write_text(req_hash)
out = subprocess.run([str(EVAL_PY), '-c',
    'import sys,torch,vllm,transformers,datasets,qwen_vl_utils,importlib.metadata as m;'
    'print(sys.version.split()[0], torch.__version__, vllm.__version__, transformers.__version__, '
    'datasets.__version__, m.version("qwen-vl-utils"), torch.cuda.is_available())'],
    capture_output=True, text=True, check=True).stdout.strip()
print('평가 환경 (python torch vllm transformers datasets qwen-vl-utils cuda):', out)
pyv, torchv, vllmv, tfv, dsv, qvlv, cuda = out.split()
assert (torchv.split('+')[0], vllmv, tfv, dsv, qvlv) == ('2.8.0', '0.11.0', '4.57.1', '4.2.0', '0.0.14'), 'baseline 실행 환경과 패키지 버전이 다릅니다'
assert cuda == 'True', '평가 환경에서 CUDA를 찾지 못했습니다'
print('✔ baseline과 같은 패키지 버전')

$ /usr/bin/python3 -m pip install -q uv
$ /usr/bin/python3 -m uv venv --python 3.12 /content/mmmu_eval_venv
requirements:
# Candidate pins; NOT yet GPU-validated. See README and Colab verification.
# After successful A100 smoke/resume tests, capture the installed environment
# using scripts/freeze_verified_environment.py and review/commit the lock.
vllm==0.11.0
torch==2.8.0
torchvision==0.23.0
torchaudio==2.8.0
transformers==4.57.1
qwen-vl-utils==0.0.14
datasets==4.2.0
Pillow==11.3.0
PyYAML==6.0.3
pytest==8.4.2

$ /usr/bin/python3 -m uv pip install --python /content/mmmu_eval_venv/bin/python -r /content/drive/MyDrive/MMDL/code/qwen3_vl_mmmu_eval/requirements.txt
평가 환경 (python torch vllm transformers datasets qwen-vl-utils cuda): 3.12.3 2.8.0+cu128 0.11.0 4.57.1 4.2.0 0.0.14 True
✔ baseline과 같은 패키지 버전


## 3. A100 확인 및 adapter 병합

- 40GB A100인지 확인합니다.
- handoff의 adapter를 base 4B(revision 고정)에 병합해 Colab 로컬에 저장합니다. 학습 노트북의 `merge.py`와 같은 방식(CPU, bf16, `safe_merge`, 원본 processor 저장)입니다. 병합에는 평가 가상환경에 `peft==0.17.1`·`accelerate==1.11.0`을 의존성 없이 추가하고, 추가 후 평가 패키지 버전이 그대로인지 다시 확인합니다.
- 가중치 파일이 `model.safetensors.index.json`과 맞는지, chat template이 baseline 실행 때와 같은지 확인합니다.


In [4]:
import torch
assert torch.cuda.is_available(), 'GPU 런타임을 선택하세요.'
gpu_name = torch.cuda.get_device_name(0)
assert gpu_name == REQUIRED_GPU, (f'이 재평가는 {REQUIRED_GPU}에서만 돌립니다. 현재: {gpu_name}\n'
    '→ 런타임 > 런타임 연결 해제 및 삭제 후 다시 연결해 40GB를 받으세요.')
print('GPU:', gpu_name)

def check_weights(model_dir, fix=False):
    files = {p.name for p in model_dir.glob('*.safetensors')}
    assert files, f'*.safetensors 가중치 없음: {model_dir}'
    index = model_dir/'model.safetensors.index.json'
    if not index.is_file():
        assert files == {'model.safetensors'}, f'index 없이 여러 shard: {files}'
        return
    need = set(json.loads(index.read_text())['weight_map'].values())
    missing = need - files
    if missing and fix:
        for name in sorted(missing):
            stem = name[:-len('.safetensors')]
            alt = [f for f in files if re.fullmatch(re.escape(stem) + r'[-_ ]?\(?\d{1,3}\)?\.safetensors', f)]
            assert len(alt) == 1, f'{name}에 해당하는 파일을 하나로 특정할 수 없음: {alt}'
            print(f'  파일명 복구: {alt[0]} -> {name}')
            (model_dir/alt[0]).rename(model_dir/name)
        files = {p.name for p in model_dir.glob('*.safetensors')}
        missing = need - files
    assert not missing, f'index가 요구하는 가중치 파일 없음: {missing} (있는 파일: {sorted(files)})'
    extra = files - need
    assert not extra, f'index에 없는 가중치 파일이 섞여 있음: {extra}'

MERGE_PY = Path('/content/grpo_merge.py')
MERGE_PY.write_text('''import json, os, sys, torch
from pathlib import Path
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
from peft import PeftModel
adapter, out, model_id, rev, info = sys.argv[1], Path(sys.argv[2]), sys.argv[3], sys.argv[4], json.loads(sys.argv[5])
tmp = out.with_name(out.name + '.building')
base = Qwen3VLForConditionalGeneration.from_pretrained(model_id, revision=rev, dtype=torch.bfloat16, device_map={'': 'cpu'}, low_cpu_mem_usage=True)
merged = PeftModel.from_pretrained(base, adapter, is_trainable=False).merge_and_unload(safe_merge=True)
merged.config.use_cache = True
merged.save_pretrained(tmp, safe_serialization=True, max_shard_size='4GB')
AutoProcessor.from_pretrained(model_id, revision=rev).save_pretrained(tmp)
(tmp / 'merge_info.json').write_text(json.dumps(info, indent=2))
os.replace(tmp, out)
print('merged:', out)
''')
merge_info = {'base': BASE_MODEL_ID, 'base_revision': BASE_REVISION, 'adapter': str(ADAPTER_DIR), 'policy_id': POLICY_ID,
              'method': 'virl39k_round_grpo', 'run_tag': GRPO_RUN_TAG, 'selected_name': SELECTED_NAME,
              'handoff_sha256': _sha(HANDOFF_FILE)}
info = merge_info
if (LOCAL_MODEL/'merge_info.json').is_file():
    old = json.loads((LOCAL_MODEL/'merge_info.json').read_text())
    assert old['policy_id'] == POLICY_ID, f'로컬 병합 모델이 다른 adapter입니다: {old}'
    print('이미 병합된 로컬 모델 재사용:', LOCAL_MODEL)
else:
    shutil.rmtree(LOCAL_MODEL.with_name(LOCAL_MODEL.name + '.building'), ignore_errors=True)
    assert shutil.disk_usage('/content').free > 20 * 2**30, '로컬 디스크 여유 공간 부족(병합에 약 9GB 필요)'
    checked([sys.executable, '-m', 'uv', 'pip', 'install', '--python', str(EVAL_PY), '--no-deps', 'peft==0.17.1', 'accelerate==1.11.0'])
    vers = subprocess.run([str(EVAL_PY), '-c', 'import torch,vllm,transformers,peft;print(torch.__version__,vllm.__version__,transformers.__version__,peft.__version__)'],
                          capture_output=True, text=True, check=True).stdout.split()
    assert (vers[0].split('+')[0], vers[1], vers[2], vers[3]) == ('2.8.0', '0.11.0', '4.57.1', '0.17.1'), f'평가 환경 버전이 바뀌었습니다: {vers}'
    t0 = time.time()
    checked([EVAL_PY, MERGE_PY, ADAPTER_DIR, LOCAL_MODEL, BASE_MODEL_ID, BASE_REVISION, json.dumps(merge_info)])
    print(f'병합 {time.time()-t0:.0f}s')
check_weights(LOCAL_MODEL, fix=True)
MODEL_FOR_EVAL = LOCAL_MODEL
print('✔ 모델 입력 경로:', MODEL_FOR_EVAL)

# chat template이 baseline 실행 때와 같은지 (평가 코드는 processor.chat_template을 그대로 사용)
tmpl = subprocess.run([str(EVAL_PY), '-c',
    'import sys;from transformers import AutoProcessor;'
    'sys.stdout.write(AutoProcessor.from_pretrained(sys.argv[1]).chat_template)', str(MODEL_FOR_EVAL)],
    capture_output=True, text=True, check=True).stdout
if BASELINE_RUN_DIR is not None and (BASELINE_RUN_DIR/'chat_template.jinja').is_file():
    assert tmpl == (BASELINE_RUN_DIR/'chat_template.jinja').read_text(encoding='utf-8'), \
        '병합 모델의 chat template이 baseline과 다릅니다. 학습 노트북의 병합 셀(원본 processor 저장)을 확인하세요.'
    print('✔ chat template = baseline (sha256', hashlib.sha256(tmpl.encode()).hexdigest()[:16] + ')')
else:
    print('⚠ baseline chat_template.jinja가 없어 비교하지 못했습니다. 생성 후 프롬프트 비교로 확인합니다.')

GPU: NVIDIA A100-SXM4-40GB
$ /usr/bin/python3 -m uv pip install --python /content/mmmu_eval_venv/bin/python --no-deps peft==0.17.1 accelerate==1.11.0
$ /content/mmmu_eval_venv/bin/python /content/grpo_merge.py /content/drive/MyDrive/VIRL39K_RFT_4B/round_grpo_v4/round_02/train/checkpoint_00680_00170 /content/virl39k_grpo_round_grpo_v4_round_02_step170_55922b8dfc29_merged_for_eval Qwen/Qwen3-VL-4B-Instruct ebb281ec70b05090aa6165b016eac8ec08e71b17 {"base": "Qwen/Qwen3-VL-4B-Instruct", "base_revision": "ebb281ec70b05090aa6165b016eac8ec08e71b17", "adapter": "/content/drive/MyDrive/VIRL39K_RFT_4B/round_grpo_v4/round_02/train/checkpoint_00680_00170", "policy_id": "55922b8dfc2948d12f6e5190f02be0dd92804aa2ba0b78a19d2a3d533bcfc442", "method": "virl39k_round_grpo", "run_tag": "round_grpo_v4", "selected_name": "round_02_step170", "handoff_sha256": "29da546b57531af2903b99d106434a99b2e48d733e33bab6b78c6047318667c3"}
병합 64s
✔ 모델 입력 경로: /content/virl39k_grpo_round_grpo_v4_round_02_step170_55922b8dfc29

## 4. `final_16k.yaml` 평가 조건 검증

아래 값이 기존 63.89% Baseline과 같아야 공정한 비교입니다. YAML은 수정하지 않습니다.

In [5]:
import yaml
config = yaml.safe_load(CONFIG.read_text(encoding='utf-8'))
assert isinstance(config, dict)

def all_values(obj, name):
    found = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k == name: found.append(v)
            found.extend(all_values(v, name))
    elif isinstance(obj, list):
        for v in obj: found.extend(all_values(v, name))
    return found

expected = {'max_model_len': 24576, 'max_new_tokens': 16384, 'temperature': 0.7, 'top_p': 0.8, 'top_k': 20,
            'presence_penalty': 1.5, 'repetition_penalty': 1.0, 'min_pixels': 1310720, 'max_pixels': 5242880}
for k, v in expected.items():
    values = all_values(config, k)
    if not values:
        print(f'⚠ YAML에 {k}가 직접 없음: 실행 후 resolved_config.json에서 확인')
    else:
        assert all(x == v for x in values), f'{k}: baseline {v}와 불일치: {values}'
        print('✔', k, '=', values)
print(f'\n✔ YAML 수정 없음. seed={EVAL_SEED}는 실행 명령의 --seed로 지정합니다' + (' (확인 재평가)' if CONFIRM_RUN else '') + '.')

✔ max_model_len = [24576]
✔ max_new_tokens = [16384]
✔ temperature = [0.7]
✔ top_p = [0.8]
✔ top_k = [20]
✔ presence_penalty = [1.5]
✔ repetition_penalty = [1.0]
✔ min_pixels = [1310720]
✔ max_pixels = [5242880]

✔ YAML 수정 없음. seed=3407는 실행 명령의 --seed로 지정합니다.


## 5. MMMU 900문항 생성 + v1 채점 (약 2시간)

Drive 평가 프로젝트의 `scripts/run_mmmu_eval.sh`를 그대로 실행합니다. `--model_path`만 병합 모델로 바꿉니다.

- 진행 로그는 화면에 요약해서 보여주고, 전체 로그는 `OUTPUTS_ROOT/eval_stdout.log`에 저장합니다.
- **세션이 끊기면:** 0번부터 다시 실행하세요. 같은 `ADAPTER_PCT`면 같은 출력 폴더를 쓰고, 끝난 과목은 건너뜁니다. 모델 내용·설정·코드·패키지가 바뀌면 평가 코드가 재개를 거부합니다(섞인 결과 방지).

In [6]:
if ALREADY_DONE:
    print('생성 완료된 결과가 있어 건너뜁니다:', RUN_DIR)
else:
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    OUTPUTS_ROOT.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env['PYTHON_BIN'] = str(EVAL_PY)
    env['PYTHONPATH'] = str(PROJECT/'src')
    env['VLLM_ENABLE_V1_MULTIPROCESSING'] = '0'
    env['VLLM_BATCH_INVARIANT'] = '1'
    env['PYTHONUNBUFFERED'] = '1'
    command = ['bash', str(EVAL_SH),
               '--model_path', str(MODEL_FOR_EVAL),
               '--model_revision', BASE_REVISION,     # 로컬 폴더이므로 평가 코드가 무시함 (이전 노트북과 동일)
               '--data_root', str(DATA_ROOT),
               '--config', str(CONFIG),
               '--seed', str(EVAL_SEED),
               '--output_dir', str(RUN_DIR),
               '--require_gpu', 'A100-SXM4-40GB',
               '--stage', 'all']
    manifest = {'method': f'ViRL39K round GRPO 4B LoRA ({GRPO_RUN_TAG}, {SELECTED_NAME}) merged / MMMU 900 validation / A100 40GB',
                'required_gpu': REQUIRED_GPU, 'variant': VARIANT, 'submission_model_40gb_dir': str(PREV_RUN_DIR),
                'date_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
                'evaluation_code_location': str(PROJECT), 'source_adapter': str(ADAPTER_DIR), 'policy_id': POLICY_ID, 'handoff': handoff,
                'merge_info': info, 'model_eval_path': str(MODEL_FOR_EVAL), 'base_revision': BASE_REVISION,
                'dataset': 'MMMU/MMMU validation (900), revision 98e6ac0cb9b7b2cd2c991b85a50762edc4aedc68',
                'seed': EVAL_SEED, 'confirm_run': CONFIRM_RUN, 'baseline_v2_correct': 575, 'baseline_v2_n': 900, 'eval_env': out,
                'config_sha256': hashlib.sha256(CONFIG.read_bytes()).hexdigest(),
                'script_sha256': hashlib.sha256(EVAL_SH.read_bytes()).hexdigest(),
                'rescore_script_sha256': hashlib.sha256(RESCORE_PY.read_bytes()).hexdigest(),
                'command': command, 'resume': RESUME}
    mf = OUTPUTS_ROOT/'launch_manifest.json'
    if mf.is_file():   # 재개 시 이전 실행 기록 보존
        mf.rename(OUTPUTS_ROOT/f'launch_manifest.{int(time.time())}.json')
    mf.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

    progress = re.compile(r'\d+%\|')   # tqdm 진행 막대는 60초에 한 번만 표시
    last_bar = 0.0
    t0 = time.time()
    print('=== 900문항 추론 시작 ===', flush=True)
    with open(OUTPUTS_ROOT/'eval_stdout.log', 'a', encoding='utf-8') as log:
        log.write(f'\n===== start {datetime.datetime.now().isoformat()} =====\n')
        proc = subprocess.Popen([str(c) for c in command], cwd=PROJECT, env=env, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            log.write(line)
            if progress.search(line):
                if time.time() - last_bar < 60: continue
                last_bar = time.time()
            print(line, end='', flush=True)
        rc = proc.wait()
    print(f'\n소요 {(time.time()-t0)/60:.1f}분, 종료 코드 {rc}')
    if rc != 0:
        meta = RUN_DIR/'run_meta.json'
        if meta.is_file():
            print('run_meta.json last_error:', json.loads(meta.read_text()).get('last_error'))
        raise RuntimeError(f'평가 실패 (exit {rc}). 전체 로그: {OUTPUTS_ROOT/"eval_stdout.log"}')
    print('✔ v1 생성·채점 완료:', RUN_DIR)

=== 900문항 추론 시작 ===
/content/drive/MyDrive/MMDL/code/qwen3_vl_mmmu_eval/src/mmmu_eval/cli.py:91: UserWarning: Local model directory: ignoring --model_revision
  warnings.warn("Local model directory: ignoring --model_revision")
GPU: NVIDIA A100-SXM4-40GB
Accounting: 30
Agriculture: 30
Architecture_and_Engineering: 30
Art: 30
Art_Theory: 30
Basic_Medical_Science: 30
Biology: 30
Chemistry: 30
Clinical_Medicine: 30
Computer_Science: 30
Design: 30
Diagnostics_and_Laboratory_Medicine: 30
Economics: 30
Electronics: 30
Energy_and_Power: 30
Finance: 30
Geography: 30
History: 30
Literature: 30
Manage: 30
Marketing: 30
Materials: 30
Math: 30
Mechanical_Engineering: 30
Music: 30
Pharmacy: 30
Physics: 30
Psychology: 30
Public_Health: 30
Sociology: 30
Dataset counts: {'per_subject': {'Accounting': 30, 'Agriculture': 30, 'Architecture_and_Engineering': 30, 'Art': 30, 'Art_Theory': 30, 'Basic_Medical_Science': 30, 'Biology': 30, 'Chemistry': 30, 'Clinical_Medicine': 30, 'Computer_Science': 30, 'Design

## 5-1. 생성 결과 검사

- 정확히 900문항인지, 실제 실행 설정(`resolved_config.json`)이 baseline 조건과 같은지 확인합니다.
- **900문항 모두 프롬프트(`prompt_text`)와 입력 토큰 수가 baseline과 같은지** 확인합니다. 같으면 모델 가중치만 바뀐 공정한 비교입니다.
- 이 단계의 `scores.json`은 **v1 채점**이라 baseline v2 63.89%와 직접 비교하지 않습니다.

In [7]:
raw, scored, score_v1, resolved = (RUN_DIR/'raw_outputs.jsonl', RUN_DIR/'scored.jsonl',
                                   RUN_DIR/'scores.json', RUN_DIR/'resolved_config.json')
for f in (raw, scored, score_v1, resolved):
    assert f.is_file(), f'평가 완료 파일 누락: {f}'
raw_rows = [json.loads(l) for l in open(raw, encoding='utf-8') if l.strip()]
assert len(raw_rows) == 900, f'900문항 전부 생성되지 않음: {len(raw_rows)}'
print('✔ 원문 응답', len(raw_rows), '개')
gpus = {r.get('gpu') for r in raw_rows}
assert gpus == {REQUIRED_GPU}, f'900문항 중 40GB가 아닌 GPU로 생성된 문항이 있습니다: {gpus}'
print('✔ 900문항 모두', REQUIRED_GPU, '에서 생성')
rc = json.loads(resolved.read_text(encoding='utf-8'))
for name, v in expected.items():
    found = all_values(rc, name)
    assert all(x == v for x in found), f'실제 실행 설정 {name} 불일치: {found}'
print('✔ resolved_config.json 설정 = baseline 조건')
print('v1 scores.json (참고용):', score_v1.read_text(encoding='utf-8')[:1500])

if BASELINE_RUN_DIR is not None:
    base_rows = {r['id']: r for r in map(json.loads, open(BASELINE_RUN_DIR/'scored.jsonl', encoding='utf-8'))}
    new_rows = {r['id']: r for r in raw_rows}
    assert set(base_rows) == set(new_rows), '문항 ID 집합이 baseline과 다릅니다'
    diff_prompt = [i for i in new_rows if new_rows[i]['prompt_text'] != base_rows[i]['prompt_text']]
    diff_input = [i for i in new_rows if new_rows[i]['input_tokens'] != base_rows[i]['input_tokens']]
    print(f'프롬프트 불일치 {len(diff_prompt)}건, 입력 토큰 수 불일치 {len(diff_input)}건')
    assert not diff_prompt and not diff_input, f'baseline과 입력이 다릅니다: {(diff_prompt+diff_input)[:5]}'
    print('✔ 900문항 모두 프롬프트·이미지 입력이 baseline과 동일')

✔ 원문 응답 900 개
✔ 900문항 모두 NVIDIA A100-SXM4-40GB 에서 생성
✔ resolved_config.json 설정 = baseline 조건
v1 scores.json (참고용): {
  "subjects": [
    {
      "subject": "Accounting",
      "data_num": 30,
      "correct": 19,
      "accuracy": 0.6333333333333333,
      "mean_output_tokens": 3503.3333333333335
    },
    {
      "subject": "Agriculture",
      "data_num": 30,
      "correct": 13,
      "accuracy": 0.43333333333333335,
      "mean_output_tokens": 166.8
    },
    {
      "subject": "Architecture_and_Engineering",
      "data_num": 30,
      "correct": 16,
      "accuracy": 0.5333333333333333,
      "mean_output_tokens": 8194.7
    },
    {
      "subject": "Art",
      "data_num": 30,
      "correct": 14,
      "accuracy": 0.4666666666666667,
      "mean_output_tokens": 435.03333333333336
    },
    {
      "subject": "Art_Theory",
      "data_num": 30,
      "correct": 21,
      "accuracy": 0.7,
      "mean_output_tokens": 401.6
    },
    {
      "subject": "Basic_Medical_Science",

## 6. v2 재채점 (GPU 생성 없음)

`raw_outputs.jsonl`을 기존 `rescore.py`로 재채점합니다. 파서 규칙은 바꾸지 않습니다. 이번 실험만 들어 있는 `OUTPUTS_ROOT`를 넘기므로 다른 실험과 섞이지 않습니다. 결정적 계산이라 다시 실행해도 결과가 같습니다.

In [8]:
V2_REPORT_DIR.mkdir(parents=True, exist_ok=True)
env_score = os.environ.copy()
env_score['PYTHONPATH'] = str(PROJECT/'src')
checked([sys.executable, str(RESCORE_PY), '--outputs-dir', str(OUTPUTS_ROOT),
         '--out-dir', str(V2_REPORT_DIR), '--runs', RUN_NAME], cwd=PROJECT, env=env_score)
print('✔ v2 재채점 리포트:', V2_REPORT_DIR)

$ /usr/bin/python3 /content/drive/MyDrive/MMDL/code/qwen3_vl_mmmu_eval/scripts/rescore.py --outputs-dir /content/drive/MyDrive/MMDL/pmc_vqa/eval/mmmu_eval_outputs/virl39k_grpo_round_grpo_v4_round_02_step170_seed3407_16k_a100_40gb --out-dir /content/drive/MyDrive/MMDL/pmc_vqa/eval/mmmu_eval_outputs/virl39k_grpo_round_grpo_v4_round_02_step170_seed3407_16k_a100_40gb_v2 --runs lora_merged
✔ v2 재채점 리포트: /content/drive/MyDrive/MMDL/pmc_vqa/eval/mmmu_eval_outputs/virl39k_grpo_round_grpo_v4_round_02_step170_seed3407_16k_a100_40gb_v2


## 7. Baseline 비교 분석

v2 파서로 baseline·이번 모델·(있으면) 이전 RFT 100% 결과(63.78%)를 같은 방식으로 다시 채점해 비교합니다. baseline 재계산이 575/900이 아니면 멈춥니다.

- **전체:** 정확도, 객관식/주관식, 평균·중앙 출력 토큰, 잘림률, unparsed 비율
- **학습 형식 출현:** `Evidence:`로 시작하는 응답 수 등. MMMU에서도 학습 형식을 쓰는지 봅니다.
- **`Final Answer` vs 파서:** 학습 형식의 마지막 줄 문자와 v2 파서가 뽑은 답이 다른 건수. 많으면 파서가 Reasoning 속 다른 보기를 잡은 것입니다(진단용, 공식 수치는 v2만 사용).
- **문항별 변화:** baseline 대비 새로 맞힘/새로 틀림, McNemar 검정
- **과목군·과목별:** 의료 5과목, Math, Physics, 추론형 6과목, 30과목 전체(과목별 평균 토큰 포함)

결과는 `V2_REPORT_DIR/comparison_vs_baseline.json`, `per_question_vs_baseline.csv`로 저장됩니다.

In [9]:
import re, json, csv, random, statistics
from math import comb
from collections import Counter, defaultdict
if str(PROJECT/'src') not in sys.path: sys.path.insert(0, str(PROJECT/'src'))
from mmmu_eval.parse import DEFAULT_REGEX
from mmmu_eval.parse_v2 import extract_answer_v2

S_prev_expected = 569   # 제출 모델 40GB v2
MED5 = ['Basic_Medical_Science','Clinical_Medicine','Diagnostics_and_Laboratory_Medicine','Pharmacy','Public_Health']
REASONING_SUBJ = ['Math','Physics','Chemistry','Electronics','Energy_and_Power','Mechanical_Engineering']
# 학습 타겟 형식의 마지막 줄 "Final Answer: X" (대소문자 구분 문자만)
FA_RE = re.compile(r'[Ff]inal\s*[Aa]nswer\s*(?:is\s*)?[:：]?\s*\**\s*\(?\s*([A-I])(?![A-Za-z0-9])')

def load_v2(run_dir):
    rows = [json.loads(l) for l in open(run_dir/'scored.jsonl', encoding='utf-8') if l.strip()]
    cfg = json.loads((run_dir/'resolved_config.json').read_text(encoding='utf-8'))
    regex = cfg['parser']['regex'] if 'parser' in cfg else DEFAULT_REGEX
    out = {}
    for r in rows:
        p = extract_answer_v2(r['response'], dict(r['scoring_choices']), regex, r['question_type'], r['truncated'])
        ok = p['method'] not in ('unparsed','refused') and p['extracted'] == r['scoring_answer']
        out[r['id']] = {'row': r, 'v2': p, 'ok': ok}
    return out

def stats(d):
    vals = list(d.values()); n = len(vals)
    mc = [v for v in vals if v['row']['question_type'] == 'multiple-choice']
    op = [v for v in vals if v['row']['question_type'] != 'multiple-choice']
    toks = [v['row']['output_tokens'] for v in vals]
    return {'n': n, 'correct': sum(v['ok'] for v in vals), 'acc': 100*sum(v['ok'] for v in vals)/n,
            'acc_mc': 100*sum(v['ok'] for v in mc)/max(len(mc),1), 'acc_open': 100*sum(v['ok'] for v in op)/max(len(op),1),
            'mean_output_tokens': statistics.mean(toks), 'median_output_tokens': statistics.median(toks),
            'trunc_pct': 100*sum(v['row']['truncated'] for v in vals)/n,
            'unparsed_pct': 100*sum(v['v2']['method'] == 'unparsed' for v in vals)/n,
            'methods': dict(Counter(v['v2']['method'] for v in vals))}

def group_acc(d, subjects):
    vs = [v for v in d.values() if v['row']['subject'] in subjects]
    return sum(v['ok'] for v in vs), len(vs)

def mcnemar(a, b):
    gain = sum((not a[i]['ok']) and b[i]['ok'] for i in a)
    loss = sum(a[i]['ok'] and (not b[i]['ok']) for i in a)
    n, k = gain+loss, min(gain, loss)
    p = min(1.0, 2*sum(comb(n, j) for j in range(k+1))/2**n) if n else 1.0
    return gain, loss, p

new = load_v2(RUN_DIR)
assert len(new) == 900
runs = {'new': new}
if BASELINE_RUN_DIR is not None:
    runs['baseline'] = load_v2(BASELINE_RUN_DIR)
    assert stats(runs['baseline'])['correct'] == 575, 'baseline v2 재계산이 575/900이 아닙니다. baseline 폴더를 확인하세요.'
    assert set(runs['baseline']) == set(new)
if PREV_RUN_DIR is not None and (PREV_RUN_DIR/'scored.jsonl').is_file():
    runs['prev_rft'] = load_v2(PREV_RUN_DIR)   # 이 노트북에서는 제출 모델(자기 풀이 학습)의 40GB 평가
    assert S_prev_expected == stats(runs['prev_rft'])['correct'], '제출 모델 40GB 결과가 569/900이 아닙니다. PREV_RUN_DIR를 확인하세요.'
S = {k: stats(v) for k, v in runs.items()}

print('=== 전체 (v2 파서) ===')
print(f"{'':16}{'정확도':>10}{'객관식':>9}{'주관식':>9}{'평균토큰':>10}{'중앙토큰':>9}{'잘림%':>8}{'unparsed%':>11}")
for k, s in S.items():
    print(f"{k:16}{s['correct']:>4}/900 {s['acc']:5.2f}%{s['acc_mc']:8.2f}{s['acc_open']:9.2f}"
          f"{s['mean_output_tokens']:10.1f}{s['median_output_tokens']:9.0f}{s['trunc_pct']:8.2f}{s['unparsed_pct']:11.2f}")
print('v2 method 분포 (new):', S['new']['methods'])

# 학습 형식이 MMMU 출력에 얼마나 나타나는지
resp = {i: v['row']['response'] for i, v in new.items()}
fmt = {'starts_with_Evidence': sum(r.lstrip().startswith('Evidence:') for r in resp.values()),
       'has_Reasoning': sum('Reasoning:' in r for r in resp.values()),
       'has_Final_Answer': sum(bool(FA_RE.search(r)) for r in resp.values())}
print('\n=== 학습 형식 출현 (900문항 중) ===', fmt)

# Final Answer 문자 vs 파서 추출 결과 (객관식, 잘리지 않은 응답)
fa_rows = []
for i, v in new.items():
    r = v['row']
    if r['question_type'] != 'multiple-choice' or r['truncated']: continue
    hits = [m.group(1) for m in FA_RE.finditer(r['response']) if m.group(1) in r['scoring_choices']]
    if hits: fa_rows.append((i, hits[-1], v['v2']['extracted'], r['scoring_answer']))
mis = [x for x in fa_rows if x[1] != x[2]]
fa_correct = sum(fa == gold for _, fa, _, gold in fa_rows)
mis_fa_right = sum(fa == gold for _, fa, ext, gold in mis)
mis_parser_right = sum(ext == gold for _, fa, ext, gold in mis)
print(f"\n=== Final Answer vs v2 파서 (객관식·잘림 없음, Final Answer 있는 {len(fa_rows)}건) ===")
print(f"불일치 {len(mis)}건 ({100*len(mis)/max(len(fa_rows),1):.1f}%) | 그중 Final Answer가 정답 {mis_fa_right}건, 파서 결과가 정답 {mis_parser_right}건")
print(f"(진단용) Final Answer 문자 기준 정답 {fa_correct}/{len(fa_rows)}. 공식 수치는 v2 파서 결과만 사용합니다.")

if 'baseline' in runs:
    base = runs['baseline']
    g, l, p = mcnemar(base, new)
    print(f"\n=== baseline → new 문항별 변화 === 새로 맞힘 {g} / 새로 틀림 {l} / 순변화 {g-l:+d} | McNemar p={p:.4f}")
    if 'prev_rft' in runs:
        g2, l2, p2 = mcnemar(runs['prev_rft'], new)
        print(f"제출 모델(40GB, {S['prev_rft']['correct']}) → new: 새로 맞힘 {g2} / 새로 틀림 {l2} | p={p2:.4f}")
    print('\n=== 과목군 ===')
    for name, subs in (('의료 5과목', MED5), ('Math', ['Math']), ('Physics', ['Physics']), ('추론형 6과목', REASONING_SUBJ)):
        line = f'{name:12}'
        for k, d in runs.items():
            c, n = group_acc(d, subs); line += f'  {k} {c}/{n} ({100*c/n:.1f}%)'
        print(line)
    print('\n=== 과목별 (Δ 오름차순) ===')
    subj_rows = []
    for subj in sorted({v['row']['subject'] for v in new.values()}):
        bc, n = group_acc(base, [subj]); nc, _ = group_acc(new, [subj])
        bt = statistics.mean(v['row']['output_tokens'] for v in base.values() if v['row']['subject'] == subj)
        nt = statistics.mean(v['row']['output_tokens'] for v in new.values() if v['row']['subject'] == subj)
        subj_rows.append({'subject': subj, 'baseline': bc, 'new': nc, 'n': n, 'delta': nc-bc,
                          'baseline_tokens': round(bt, 1), 'new_tokens': round(nt, 1)})
    print(f"{'subject':38}{'base':>6}{'new':>6}{'Δ':>5}{'base tok':>10}{'new tok':>9}")
    for s in sorted(subj_rows, key=lambda x: x['delta']):
        print(f"{s['subject']:38}{s['baseline']:>6}{s['new']:>6}{s['delta']:>+5}{s['baseline_tokens']:>10.0f}{s['new_tokens']:>9.0f}")
else:
    g = l = p = None; subj_rows = []
    print('\n⚠ baseline scored.jsonl이 없어 문항별·과목별 비교를 건너뜁니다 (BASELINE_RUN_DIR 설정 확인).')

report = {'experiment': EXPERIMENT_NAME, 'variant': VARIANT, 'adapter_pct': ADAPTER_PCT, 'runs': S, 'format': fmt,
          'final_answer_vs_parser': {'n_with_final_answer': len(fa_rows), 'mismatch': len(mis),
                                     'mismatch_final_answer_correct': mis_fa_right, 'mismatch_parser_correct': mis_parser_right,
                                     'final_answer_letter_correct': fa_correct},
          'paired_vs_baseline': {'gain': g, 'loss': l, 'p': p}, 'per_subject': subj_rows}
V2_REPORT_DIR.mkdir(parents=True, exist_ok=True)
(V2_REPORT_DIR/'comparison_vs_baseline.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
with open(V2_REPORT_DIR/'per_question_vs_baseline.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f); w.writerow(['id','subject','question_type','gold','new_extracted','new_method','new_correct',
                                   'baseline_correct','new_output_tokens','new_truncated'])
    for i, v in new.items():
        r = v['row']
        w.writerow([i, r['subject'], r['question_type'], r['scoring_answer'], v['v2']['extracted'], v['v2']['method'],
                    int(v['ok']), int(runs['baseline'][i]['ok']) if 'baseline' in runs else '', r['output_tokens'], int(r['truncated'])])
print('\n저장:', V2_REPORT_DIR/'comparison_vs_baseline.json', '|', V2_REPORT_DIR/'per_question_vs_baseline.csv')

=== 전체 (v2 파서) ===
                       정확도      객관식      주관식      평균토큰     중앙토큰     잘림%  unparsed%
new              596/900 66.22%   67.30    49.06    2122.4      456    2.44       3.89
baseline         575/900 63.89%   64.94    47.17    2780.9      480    9.11       8.78
prev_rft         569/900 63.22%   64.46    43.40    2364.4      442    5.89       6.33
v2 method 분포 (new): {'fallback': 208, 'regex': 281, 'option': 192, 'text': 184, 'unparsed': 35}

=== 학습 형식 출현 (900문항 중) === {'starts_with_Evidence': 0, 'has_Reasoning': 9, 'has_Final_Answer': 378}

=== Final Answer vs v2 파서 (객관식·잘림 없음, Final Answer 있는 378건) ===
불일치 9건 (2.4%) | 그중 Final Answer가 정답 4건, 파서 결과가 정답 0건
(진단용) Final Answer 문자 기준 정답 251/378. 공식 수치는 v2 파서 결과만 사용합니다.

=== baseline → new 문항별 변화 === 새로 맞힘 74 / 새로 틀림 53 / 순변화 +21 | McNemar p=0.0755
제출 모델(40GB, 569) → new: 새로 맞힘 81 / 새로 틀림 54 | p=0.0249

=== 과목군 ===
의료 5과목        new 105/150 (70.0%)  baseline 104/150 (69.3%)  prev_rft 100/150 (66.7%)
Math          new 22/30 (73

In [10]:
# 7-0. baseline 출력 길이 구간별 · 과목군별 정확도 변화 (v2 파서, runs['baseline'] vs runs['new'])
import math
from math import comb
if 'baseline' in runs:
    base = runs['baseline']
    ids = list(new.keys())
    def tok(i): return base[i]['row']['output_tokens']
    buckets = [('<300',0,300),('300-800',300,800),('800-2k',800,2000),('2k-5k',2000,5000),('>5k (반복·잘림)',5000,10**9)]
    def sign_p(g, l):
        n = g + l
        return 1.0 if n == 0 else min(1.0, 2*sum(comb(n, k) for k in range(min(g, l)+1))/2**n)
    def row(name, sel):
        b = sum(base[i]['ok'] for i in sel); a = sum(new[i]['ok'] for i in sel)
        g = sum(new[i]['ok'] and not base[i]['ok'] for i in sel); l = sum(base[i]['ok'] and not new[i]['ok'] for i in sel)
        nt = statistics.median(new[i]['row']['output_tokens'] for i in sel) if sel else 0
        print(f"{name:34s} n={len(sel):3d}  baseline {b:3d} ({100*b/max(len(sel),1):5.1f}%)  new {a:3d} ({100*a/max(len(sel),1):5.1f}%)  Δ{a-b:+d} (+{g}/-{l}) p={sign_p(g,l):.2f}  새 모델 중앙토큰 {nt:.0f}")
    print('=== baseline 출력 길이 구간별 ===')
    for name, lo, hi in buckets: row(name, [i for i in ids if lo <= tok(i) < hi])
    groups = {
      '약점 STEM(수학·화학·재료·공학계열)': ['Math','Chemistry','Materials','Mechanical_Engineering','Architecture_and_Engineering','Energy_and_Power','Electronics','Computer_Science','Physics'],
      '비즈니스(회계·금융·경제·경영·마케팅)': ['Accounting','Finance','Economics','Manage','Marketing'],
      '의료 5과목': MED5,
      '미술·음악·디자인': ['Art','Art_Theory','Design','Music'],
      '그 외': None}
    used = set()
    print('\n=== 과목군별 ===')
    for name, subs in groups.items():
        if subs is None: sel = [i for i in ids if new[i]['row']['subject'] not in used]
        else:
            sel = [i for i in ids if new[i]['row']['subject'] in subs]; used |= set(subs)
        row(name, sel)
else:
    print('baseline 결과가 없어 건너뜁니다.')


=== baseline 출력 길이 구간별 ===
<300                               n=301  baseline 235 ( 78.1%)  new 231 ( 76.7%)  Δ-4 (+3/-7) p=0.34  새 모델 중앙토큰 210
300-800                            n=255  baseline 165 ( 64.7%)  new 164 ( 64.3%)  Δ-1 (+15/-16) p=1.00  새 모델 중앙토큰 418
800-2k                             n=113  baseline  79 ( 69.9%)  new  80 ( 70.8%)  Δ+1 (+9/-8) p=1.00  새 모델 중앙토큰 1170
2k-5k                              n= 84  baseline  54 ( 64.3%)  new  52 ( 61.9%)  Δ-2 (+8/-10) p=0.81  새 모델 중앙토큰 2784
>5k (반복·잘림)                        n=147  baseline  42 ( 28.6%)  new  69 ( 46.9%)  Δ+27 (+39/-12) p=0.00  새 모델 중앙토큰 7577

=== 과목군별 ===
약점 STEM(수학·화학·재료·공학계열)             n=270  baseline 151 ( 55.9%)  new 167 ( 61.9%)  Δ+16 (+39/-23) p=0.06  새 모델 중앙토큰 2342
비즈니스(회계·금융·경제·경영·마케팅)              n=150  baseline 112 ( 74.7%)  new 111 ( 74.0%)  Δ-1 (+8/-9) p=1.00  새 모델 중앙토큰 515
의료 5과목                             n=150  baseline 104 ( 69.3%)  new 105 ( 70.0%)  Δ+1 (+6/-5) p=1.00  새 모델 중앙토큰 374
미술·음악·디자인 

## 7-2. 판정과 채택 (평가 전에 고정한 기준)

c = 이번 정답 수(v2, 40GB), baseline 575.
- **첫 평가(seed 3407):** c ≥ 594(사전 지정 통과선, +19) → **확인 재평가 필요**(0번 셀 `CONFIRM_RUN = True`). 576~593 → 개선 입증 안 됨, 575 이하 → 개선 없음. 둘 다 제출 모델 유지.
- **확인 재평가(seed 42):** 두 평가 평균 ≥ 588 → **채택**, 아니면 제출 모델 유지. 재시도 없음.

baseline·제출 모델 대비 변화를 **잘림 관련 / 그 외**로 나눠 봅니다. GRPO는 base에서 시작했으므로 baseline 대비 변화가 GRPO 효과입니다. Drive에 이전 변형(A-전체·A-소량·짧은 풀이) 판정 파일이 있으면 함께 표시합니다.

결과는 `V2_REPORT_DIR/grpo_verdict.json`, `grpo_adoption.json`에 저장합니다(확인 재평가는 seed 42 폴더).

In [11]:
# 7-2. 잘림 분해 + 판정
def trunc_churn(a, b):
    k = Counter()
    for i in a:
        ao, bo = a[i]['ok'], b[i]['ok']; at, bt = bool(a[i]['row']['truncated']), bool(b[i]['row']['truncated'])
        if ao == bo: continue
        if bo: k['gain_trunc' if at and not bt else 'gain_other'] += 1
        else:  k['loss_trunc' if bt and not at else 'loss_other'] += 1
    k['resolved'] = sum(bool(a[i]['row']['truncated']) and not b[i]['row']['truncated'] for i in a)
    k['newly'] = sum(b[i]['row']['truncated'] and not bool(a[i]['row']['truncated']) for i in a)
    k['trunc_a'] = sum(bool(a[i]['row']['truncated']) for i in a); k['trunc_b'] = sum(bool(b[i]['row']['truncated']) for i in a)
    k['net_trunc'] = k['gain_trunc'] - k['loss_trunc']; k['net_other'] = k['gain_other'] - k['loss_other']
    return dict(k)
def show_churn(label, k):
    print(f"{label}\n  잘림 {k['trunc_a']} → {k['trunc_b']}건 | 해소 {k['resolved']}건(정답으로 바뀜 {k.get('gain_trunc',0)}) · 새로 잘림 {k['newly']}건(오답으로 바뀜 {k.get('loss_trunc',0)})\n"
          f"  순변화: 잘림 {k['net_trunc']:+d} / 나머지 {k['net_other']:+d} (+{k.get('gain_other',0)}/-{k.get('loss_other',0)}) / 합계 {k['net_trunc']+k['net_other']:+d}")
rec = {'variant': VARIANT, 'run_tag': GRPO_RUN_TAG, 'selected_name': SELECTED_NAME, 'policy_id': POLICY_ID, 'posthoc_candidate': bool(handoff.get('posthoc')),
       'virl_proxy_gain': handoff['proxy_gain'], 'required_gpu': REQUIRED_GPU, 'experiment': EXPERIMENT_NAME,
       'new_correct': S['new']['correct'], 'trunc_pct': S['new']['trunc_pct'], 'median_output_tokens': S['new']['median_output_tokens']}
if 'baseline' in runs:
    rec['vs_baseline'] = kb = trunc_churn(runs['baseline'], new)
    show_churn('=== baseline(40GB, 575) → 이번 (GRPO 효과: base에서 시작) ===', kb)
if 'prev_rft' in runs:
    rec['submission_model_correct'] = S['prev_rft']['correct']
    rec['submission_model_trunc_pct'] = S['prev_rft']['trunc_pct']
    rec['vs_submission_model'] = ks = trunc_churn(runs['prev_rft'], new)
    show_churn(f"\n=== 제출 모델(40GB, {S['prev_rft']['correct']}) → 이번 ===", ks)
    print(f"  잘림률: 제출 모델 {S['prev_rft']['trunc_pct']:.2f}% → 이번 {S['new']['trunc_pct']:.2f}%")
c = S['new']['correct']
if c >= 594:   v = '사전 지정 통과선 통과 (≥594)'
elif c >= 576: v = '개선 입증 안 됨 (576~593)'
else:          v = '개선 없음 (≤575)'
rec['verdict'] = v; rec['passes_594'] = c >= 594; rec['seed'] = EVAL_SEED; rec['confirm_run'] = CONFIRM_RUN
print(f'\n판정 [{VARIANT} {SELECTED_NAME}, seed {EVAL_SEED}]: {c}/900 ({100*c/900:.2f}%) → {v}')
(V2_REPORT_DIR/'grpo_verdict.json').write_text(json.dumps(rec, ensure_ascii=False, indent=2), encoding='utf-8')
print('저장:', V2_REPORT_DIR/'grpo_verdict.json')

# --- 채택 규칙 + 이전 변형 기록 (있으면) ---
score = {'grpo': c}
for name in ('full', 'small'):
    f = DRIVE_RESULTS_BASE / f'virl39k_teacher30b_{name}_100pct_seed3407_16k_a100_40gb_v2' / 'teacher_verdict.json'
    if f.is_file(): score[f'teacher_{name}'] = json.loads(f.read_text())['new_correct']
f = DRIVE_RESULTS_BASE / 'virl39k_shortest_100pct_seed3407_16k_a100_40gb_v2' / 'shortest_verdict.json'
if f.is_file(): score['shortest'] = json.loads(f.read_text())['new_correct']
KEEP = '제출 모델 유지 (ViRL 자기 풀이 학습, 2e-5)'
if not CONFIRM_RUN:
    if c >= 594:
        adopt, status = None, 'needs_confirmation'
        note = '594 이상: 사전 규칙에 따라 확인 재평가가 필요합니다. 0번 셀에서 CONFIRM_RUN = True로 바꾸고 0번 셀부터 다시 실행하세요(seed 42, 약 27유닛, 2시간 20분).'
    else:
        adopt, status, note = KEEP, 'final', '594 미만이라 확인 재평가 없이 끝냅니다.'
else:
    c1 = first_verdict['new_correct']; mean2 = (c1 + c) / 2
    score.update({'grpo_seed3407': c1, 'grpo_seed42': c, 'grpo_mean': mean2})
    score.pop('grpo', None)
    adopt = f'라운드 GRPO ({GRPO_RUN_TAG}, {SELECTED_NAME})' if mean2 >= 588 else KEEP
    status = 'final'
    note = f'두 평가 평균 {mean2:.1f} (기준 588). 참고: 학습 없는 baseline은 seed 3407 575 / seed 42 578.'
print('평가된 변형(제출 모델 이후):', score)
print('채택:', adopt or '보류 (확인 재평가 필요)', '|', note)
print('보고서: 제출 모델 이후 평가한 변형 수, 각 수치, 확인 재평가 규칙을 함께 적습니다.')
(V2_REPORT_DIR/'grpo_adoption.json').write_text(json.dumps({'scores': score, 'submission_model_correct': S.get('prev_rft', {}).get('correct'),
    'adopt': adopt, 'status': status, 'note': note, 'policy_id': POLICY_ID, 'selected_name': SELECTED_NAME,
    'posthoc_candidate': bool(handoff.get('posthoc')),
    'rule': 'first eval (seed 3407) >=594 -> confirmation eval (seed 42, same adapter); adopt only if mean of both >=588; else keep submission model; no retries (pre-registered 2026-11-19)'},
    ensure_ascii=False, indent=2), encoding='utf-8')


=== baseline(40GB, 575) → 이번 (GRPO 효과: base에서 시작) ===
  잘림 82 → 22건 | 해소 71건(정답으로 바뀜 26) · 새로 잘림 11건(오답으로 바뀜 3)
  순변화: 잘림 +23 / 나머지 -2 (+48/-50) / 합계 +21

=== 제출 모델(40GB, 569) → 이번 ===
  잘림 53 → 22건 | 해소 44건(정답으로 바뀜 15) · 새로 잘림 13건(오답으로 바뀜 4)
  순변화: 잘림 +11 / 나머지 +16 (+66/-50) / 합계 +27
  잘림률: 제출 모델 5.89% → 이번 2.44%

판정 [grpo round_02_step170, seed 3407]: 596/900 (66.22%) → 사전 지정 통과선 통과 (≥594)
저장: /content/drive/MyDrive/MMDL/pmc_vqa/eval/mmmu_eval_outputs/virl39k_grpo_round_grpo_v4_round_02_step170_seed3407_16k_a100_40gb_v2/grpo_verdict.json
평가된 변형(제출 모델 이후): {'grpo': 596, 'teacher_full': 550, 'teacher_small': 562, 'shortest': 571}
채택: 보류 (확인 재평가 필요) | 594 이상: 사전 규칙에 따라 확인 재평가가 필요합니다. 0번 셀에서 CONFIRM_RUN = True로 바꾸고 0번 셀부터 다시 실행하세요(seed 42, 약 27유닛, 2시간 20분).
보고서: 제출 모델 이후 평가한 변형 수, 각 수치, 확인 재평가 규칙을 함께 적습니다.


655

## 7-1. 응답 샘플 확인

자동 지표로는 알 수 없는 것(Evidence에 이미지에 없는 소견을 지어내는지, 추론형 문제에서 풀이가 지나치게 짧아졌는지)을 직접 보기 위한 샘플입니다. 의료·추론형 과목과 `Final Answer` 불일치 사례를 보여줍니다.

In [12]:
rng = random.Random(0)
def show(i, tag):
    v = new[i]; r = v['row']
    b = f" | baseline {'O' if runs['baseline'][i]['ok'] else 'X'}" if 'baseline' in runs else ''
    print(f"\n----- [{tag}] {i} | gold {r['scoring_answer']} | v2 {v['v2']['extracted']} ({v['v2']['method']}) "
          f"{'O' if v['ok'] else 'X'}{b} | {r['output_tokens']} tok{' | TRUNCATED' if r['truncated'] else ''}")
    t = r['response']
    print(t if len(t) <= 1500 else t[:1000] + '\n  ...[중략]...\n' + t[-400:])
for tag, subs, k in (('의료', MED5, 3), ('추론형', REASONING_SUBJ, 3)):
    ids = [i for i, v in new.items() if v['row']['subject'] in subs]
    for i in rng.sample(ids, k): show(i, tag)
for i, fa, ext, gold in mis[:3]:
    show(i, f'Final Answer {fa} ≠ 파서 {ext}')


----- [의료] validation_Pharmacy_9 | gold A | v2 A (fallback) O | baseline O | 156 tok
The correct answer is **A. a**.

### Explanation:
In picture (a), the jar is upright and contains all the colored balls neatly packed inside, suggesting a **structured, organized state**. The system is in a stable, contained arrangement.

In picture (b), the jar is tipped over, and the balls have spilled out onto the table and floor, scattering randomly. This represents **disorder**, as the balls are no longer confined and are spread out unpredictably.

In physics and systems theory, “order” often refers to the degree of structure or predictability in a system. Picture (a) clearly shows more order than picture (b).

Thus, **(a) appears to have more order**.

✅ Final Answer: **A. a**

----- [의료] validation_Pharmacy_18 | gold A | v2 A (fallback) O | baseline X | 329 tok
The question asks to select the graph that represents **transdermal absorption by active transport**.

Let’s analyze the graphs:

- **G

## 8. 결과 해석과 다음 단계

1. **공식 비교 수치:** baseline **575/900 (63.89%, 40GB)**. 제출 모델은 569/900(40GB).
2. **7-2의 판정·채택**을 그대로 따릅니다. 노이즈 범위의 차이를 개선이라고 쓰지 않습니다.
3. **GRPO 효과:** base에서 시작했으므로 'baseline → 이번' 문항별 변화가 GRPO 효과입니다. 잘림 관련 / 그 외로 나눠 해석하고, ViRL 검증 향상(handoff의 proxy gain)이 MMMU로 이어졌는지 봅니다.
4. **결과 보존:** Drive의 `mmmu_eval_outputs/<EXPERIMENT_NAME>/`와 `<EXPERIMENT_NAME>_v2/`, 학습 폴더의 `eval_handoff.json`·`selection_report.json`을 로컬로 가져와 주세요.
5. 7-1의 응답 샘플은 MMMU 응답 원문이 출력되므로, 커밋 전에 출력을 지우세요.


In [13]:
import time
from google.colab import runtime
print('모든 셀 완료. 60초 뒤 런타임을 종료합니다. 중단하려면 셀을 정지하세요.')
time.sleep(60)
runtime.unassign()

모든 셀 완료. 60초 뒤 런타임을 종료합니다. 중단하려면 셀을 정지하세요.
